# Hands-On: MaskAnyone & Masked-Piper v2

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-org/TilburgMultiscaleSummerschool2026/blob/main/thursday-masking/notebooks/02_maskanyone_hands_on.ipynb)

**Session:** 11:00 – 12:30 · Thursday Masking Day  
**Tools covered:** MaskAnyone API, Masked-Piper v2 (MediaPipe + SAM2-tiny), masking strategies  
**Outcome:** You will mask a short video clip using three different strategies and compare visual output

---
**Concepts:**
- Person detection (YOLO / MediaPipe Pose)
- Segmentation (MediaPipe Selfie Segmentation, SAM2-tiny)
- Masking strategies: blur, pixelate, solid fill, silhouette
- COCO-17 keypoint export for downstream pose analysis

## Part 0 — Environment setup

Run this cell first. On Colab the installs take ~2 minutes.

In [ ]:
import sys, subprocess

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    subprocess.run([
        'pip', 'install', '-q',
        'opencv-python-headless',
        'mediapipe>=0.10.9',
        'numpy',
        'tqdm',
        'matplotlib',
    ], check=True)

print('Setup complete.')

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from tqdm.auto import tqdm
import tempfile, os

# ── Carbon dark theme for all plots ───────────────────────────────
plt.rcParams.update({
    'figure.facecolor':  '#161616',
    'axes.facecolor':    '#262626',
    'axes.edgecolor':    '#525252',
    'axes.labelcolor':   '#f4f4f4',
    'xtick.color':       '#c6c6c6',
    'ytick.color':       '#c6c6c6',
    'text.color':        '#f4f4f4',
    'grid.color':        '#393939',
    'grid.linestyle':    '--',
    'font.family':       'sans-serif',
})

print('Imports OK.')

---
## Part 1 — The Masked-Piper v2 Architecture

Masked-Piper v2 is a **modular pipeline** built around three swappable components:

```
Video frame
    │
    ▼
[ PoseEstimator ]   → detects body keypoints (COCO-17)
    │                 Options: MediaPipe Pose, ViTPose
    ▼
[ Segmenter ]       → produces person mask from keypoints
    │                 Options: MediaPipe Selfie, SAM2-tiny, BBox
    ▼
[ MaskingStrategy ] → applies visual effect to masked region
    │                 Options: blur, pixelate, solid, silhouette, none
    ▼
Masked frame  +  COCO-17 JSON  (→ pose analysis downstream)
```

This design means you can **mix and match**: e.g. ViTPose estimator + SAM2-tiny segmenter + blur strategy.

---
## Part 2 — Masking Strategies (code walkthrough)

Let's implement all four strategies from scratch to understand what each does to the pixel values.

In [ ]:
def apply_blur(frame: np.ndarray, mask: np.ndarray,
               kernel_size: int = 99, sigma: float = 30.0) -> np.ndarray:
    """Gaussian blur over masked region — preserves silhouette."""
    k = kernel_size | 1  # force odd
    blurred = cv2.GaussianBlur(frame, (k, k), sigma)
    alpha = cv2.merge([mask, mask, mask]) / 255.0
    return (frame * (1 - alpha) + blurred * alpha).astype(np.uint8)


def apply_pixelate(frame: np.ndarray, mask: np.ndarray,
                   block_size: int = 20) -> np.ndarray:
    """Mosaic / pixelate masked region."""
    h, w = frame.shape[:2]
    small = cv2.resize(frame, (w // block_size, h // block_size),
                       interpolation=cv2.INTER_LINEAR)
    pixelated = cv2.resize(small, (w, h), interpolation=cv2.INTER_NEAREST)
    alpha = cv2.merge([mask, mask, mask]) / 255.0
    return (frame * (1 - alpha) + pixelated * alpha).astype(np.uint8)


def apply_solid(frame: np.ndarray, mask: np.ndarray,
                color: tuple = (0, 0, 0)) -> np.ndarray:
    """Replace masked region with a solid colour (default: black)."""
    result = frame.copy()
    result[mask > 127] = color
    return result


def apply_silhouette(frame: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Black silhouette — same as solid black, intended for skeleton overlay."""
    return apply_solid(frame, mask, color=(0, 0, 0))


STRATEGIES = {
    'blur':       apply_blur,
    'pixelate':   apply_pixelate,
    'solid':      apply_solid,
    'silhouette': apply_silhouette,
}

print('Strategies defined:', list(STRATEGIES.keys()))

---
## Part 3 — Demo image: comparing all strategies

We synthesize a test image with a simulated person region so you can see each strategy side-by-side without needing real video.

In [ ]:
# ── Build a synthetic scene ─────────────────────────────────────────
rng = np.random.default_rng(42)
H, W = 360, 640

# Background: gradient
bg = np.zeros((H, W, 3), dtype=np.uint8)
for y in range(H):
    bg[y] = [int(y / H * 80), int(y / H * 60), int(y / H * 100)]

# Foreground: textured ellipse (simulated person)
frame = bg.copy()
cv2.ellipse(frame, (W//2, H//2), (120, 160), 0, 0, 360, (120, 90, 70), -1)
noise = rng.integers(0, 40, (H, W, 3), dtype=np.uint8)
frame = cv2.add(frame, noise)
cv2.putText(frame, 'PARTICIPANT', (W//2-80, H//2),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 180, 160), 2)

# Mask: ellipse region
mask = np.zeros((H, W), dtype=np.uint8)
cv2.ellipse(mask, (W//2, H//2), (120, 160), 0, 0, 360, 255, -1)

# ── Apply all strategies and plot ──────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
labels = ['Original', 'Blur', 'Pixelate', 'Solid Fill', 'Silhouette']
results = [
    frame,
    apply_blur(frame, mask),
    apply_pixelate(frame, mask),
    apply_solid(frame, mask, color=(50, 50, 200)),
    apply_silhouette(frame, mask),
]

for ax, img, lbl in zip(axes, results, labels):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(lbl, color='#f4f4f4', fontsize=11)
    ax.axis('off')

plt.suptitle('Masking strategies compared', color='#f4f4f4', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## Part 4 — MediaPipe pose estimation + segmentation

MediaPipe Holistic gives us:
- **33 body landmarks** (3D with visibility)
- **Selfie segmentation mask** (real-time, CPU-friendly)

This is the backbone of Masked-Piper v1 and is still the default CPU backend in v2.

In [ ]:
try:
    import mediapipe as mp
    print(f'MediaPipe {mp.__version__} available ✓')
    HAS_MP = True
except ImportError:
    print('MediaPipe not installed — run: pip install mediapipe')
    HAS_MP = False

In [ ]:
def mask_frame_with_mediapipe(frame_bgr: np.ndarray,
                               strategy: str = 'blur',
                               threshold: float = 0.6
                               ) -> tuple[np.ndarray, np.ndarray | None]:
    """
    Run MediaPipe Selfie Segmentation on a single frame,
    apply the chosen masking strategy, return (masked_frame, mask).
    """
    if not HAS_MP:
        return frame_bgr, None

    mp_selfie = mp.solutions.selfie_segmentation

    with mp_selfie.SelfieSegmentation(model_selection=1) as seg:
        rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        result = seg.process(rgb)
        confidence = result.segmentation_mask  # float32, 0–1

    # Threshold to binary mask
    mask = (confidence > threshold).astype(np.uint8) * 255

    # Morphological cleanup
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    fn = STRATEGIES.get(strategy, apply_blur)
    masked = fn(frame_bgr, mask)
    return masked, mask


# ── Test on our synthetic frame ─────────────────────────────────────
if HAS_MP:
    masked_mp, mask_mp = mask_frame_with_mediapipe(frame, strategy='blur')

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, img, lbl in zip(
        axes,
        [frame, mask_mp, masked_mp],
        ['Original', 'Segmentation mask', 'Blurred result']
    ):
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if len(img.shape) == 3 else img,
                  cmap='gray' if len(img.shape) == 2 else None)
        ax.set_title(lbl, color='#f4f4f4')
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Skipped — MediaPipe not available.')

---
## Part 5 — Processing a video file

Upload your own clip (any MP4 ≤ 1920×1080, H.264 recommended) or use a sample. The cell reads frame-by-frame, applies MediaPipe segmentation + your chosen strategy, and writes `_masked.mp4`.

In [ ]:
if IN_COLAB:
    from google.colab import files
    print('Upload a short MP4 clip (< 30 s recommended):')
    uploaded = files.upload()
    INPUT_VIDEO = Path(next(iter(uploaded)))
else:
    # Local: point to your prepared clip
    INPUT_VIDEO = Path('../data/raw/sample.mp4')
    if not INPUT_VIDEO.exists():
        print(f'File not found: {INPUT_VIDEO}')
        print('Set INPUT_VIDEO to an existing MP4 path.')
    else:
        print(f'Using: {INPUT_VIDEO}')

In [ ]:
STRATEGY = 'blur'   # ← change to: 'pixelate', 'solid', 'silhouette'
THRESHOLD = 0.55    # segmentation confidence cutoff (0–1)
MAX_FRAMES = 150    # cap to keep runtime short on Colab


def process_video(input_path: Path, strategy: str = 'blur',
                  threshold: float = 0.55,
                  max_frames: int | None = None) -> Path:
    """Run Masked-Piper MediaPipe pipeline on a video file."""
    cap = cv2.VideoCapture(str(input_path))
    if not cap.isOpened():
        raise RuntimeError(f'Cannot open video: {input_path}')

    fps    = cap.get(cv2.CAP_PROP_FPS)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if max_frames:
        total = min(total, max_frames)

    out_path = input_path.parent / f'{input_path.stem}__{strategy}_masked.mp4'
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(str(out_path), fourcc, fps, (width, height))

    mp_selfie = mp.solutions.selfie_segmentation
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
    fn = STRATEGIES.get(strategy, apply_blur)

    with mp_selfie.SelfieSegmentation(model_selection=1) as seg:
        for _ in tqdm(range(total), desc=f'Processing [{strategy}]'):
            ret, frame = cap.read()
            if not ret:
                break
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            conf = seg.process(rgb).segmentation_mask
            mask = (conf > threshold).astype(np.uint8) * 255
            mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
            writer.write(fn(frame, mask))

    cap.release()
    writer.release()
    print(f'Saved → {out_path}')
    return out_path


if HAS_MP and INPUT_VIDEO.exists():
    out = process_video(INPUT_VIDEO, strategy=STRATEGY,
                        threshold=THRESHOLD, max_frames=MAX_FRAMES)
else:
    print('Skipped — set INPUT_VIDEO and ensure MediaPipe is installed.')

---
## Part 6 — MaskAnyone API (Docker)

MaskAnyone runs as a **FastAPI server** inside Docker. It uses YOLO for detection and SAM2 for segmentation — higher quality but requires a GPU.

### Start the server (one-time setup)
```bash
# Clone and start
git clone https://github.com/MaskAnyone/MaskAnyone
cd MaskAnyone
docker compose up -d   # pulls ~8 GB on first run
# UI at http://localhost:5001
# API at http://localhost:8080
```

### API quick reference

In [ ]:
import requests, json, time
from pathlib import Path

MA_BASE = 'http://localhost:8080'   # adjust if running remotely


def maskanyone_upload(video_path: Path, base_url: str = MA_BASE) -> str:
    """Upload a video to MaskAnyone; returns video_id."""
    with open(video_path, 'rb') as f:
        r = requests.post(f'{base_url}/videos', files={'file': f})
    r.raise_for_status()
    return r.json()['id']


def maskanyone_run(video_id: str,
                   strategy: str = 'blurring',
                   pose_model: str = 'mediapipe',
                   base_url: str = MA_BASE) -> str:
    """Submit a masking job; returns run_id."""
    payload = {
        'videoId': video_id,
        'maskingParams': {
            'strategy': strategy,       # blurring, pixelation, solid_fill, contours
            'poseModel': pose_model,    # mediapipe, openpose, yolopose
        }
    }
    r = requests.post(f'{base_url}/runs', json=payload)
    r.raise_for_status()
    return r.json()['id']


def maskanyone_poll(run_id: str, base_url: str = MA_BASE,
                    timeout: int = 300) -> dict:
    """Poll until the run completes; returns the run result dict."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        r = requests.get(f'{base_url}/runs/{run_id}')
        r.raise_for_status()
        data = r.json()
        status = data.get('status', '')
        print(f'  status: {status}', end='\r')
        if status == 'completed':
            return data
        if status == 'failed':
            raise RuntimeError(f'Run failed: {data}')
        time.sleep(3)
    raise TimeoutError(f'Run {run_id} did not complete in {timeout}s')


def maskanyone_download(run_id: str, out_path: Path,
                        base_url: str = MA_BASE) -> Path:
    """Download the masked video from a completed run."""
    r = requests.get(f'{base_url}/runs/{run_id}/result', stream=True)
    r.raise_for_status()
    out_path.write_bytes(r.content)
    print(f'Downloaded → {out_path}')
    return out_path


print('MaskAnyone API helpers defined.')
print('Uncomment the block below when Docker server is running.')

In [ ]:
# ── Uncomment to run against a live MaskAnyone server ──────────────
#
# video_id = maskanyone_upload(INPUT_VIDEO)
# print(f'Uploaded → video_id: {video_id}')
#
# run_id = maskanyone_run(video_id, strategy='blurring', pose_model='mediapipe')
# print(f'Run submitted → run_id: {run_id}')
#
# result = maskanyone_poll(run_id)
# print('\nCompleted:', json.dumps(result, indent=2))
#
# out_video = maskanyone_download(run_id, INPUT_VIDEO.parent / 'masked_maskanyone.mp4')
print('Cell ready — start Docker and uncomment to run.')

---
## Part 7 — Masked-Piper v2: modular pipeline code

Below is a self-contained re-implementation of the Masked-Piper v2 `process_video()` pipeline showing how the three components plug together. This runs locally without the full package installed.

In [ ]:
from dataclasses import dataclass, field

# COCO-17 skeleton connections
_SKELETON_PAIRS = [
    (0, 1), (0, 2), (1, 3), (2, 4),          # head
    (5, 6), (5, 7), (7, 9), (6, 8), (8, 10), # arms
    (5, 11), (6, 12), (11, 12),               # torso
    (11, 13), (13, 15), (12, 14), (14, 16),   # legs
]


def draw_coco17_skeleton(frame: np.ndarray,
                          keypoints: np.ndarray,   # shape (17, 3) — x, y, conf
                          color: tuple = (0, 255, 80),
                          thickness: int = 2,
                          conf_thr: float = 0.3) -> None:
    """Draw COCO-17 skeleton on frame in-place."""
    for i, j in _SKELETON_PAIRS:
        if keypoints[i, 2] > conf_thr and keypoints[j, 2] > conf_thr:
            pt1 = (int(keypoints[i, 0]), int(keypoints[i, 1]))
            pt2 = (int(keypoints[j, 0]), int(keypoints[j, 1]))
            cv2.line(frame, pt1, pt2, color, thickness)
    for idx in range(keypoints.shape[0]):
        if keypoints[idx, 2] > conf_thr:
            cv2.circle(frame, (int(keypoints[idx, 0]), int(keypoints[idx, 1])), 4, color, -1)


def mediapipe_to_coco17(landmarks, w: int, h: int) -> np.ndarray:
    """
    Convert MediaPipe 33-landmark pose to approximate COCO-17.
    MediaPipe landmark indices: 0=nose, 11=L-shoulder, 12=R-shoulder, …
    COCO-17 order: nose, L-eye, R-eye, L-ear, R-ear,
                   L-shoulder, R-shoulder, L-elbow, R-elbow,
                   L-wrist, R-wrist, L-hip, R-hip,
                   L-knee, R-knee, L-ankle, R-ankle
    """
    MP_TO_COCO = [0, 2, 5, 7, 8, 11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28]
    kps = np.zeros((17, 3), dtype=np.float32)
    for coco_idx, mp_idx in enumerate(MP_TO_COCO):
        lm = landmarks.landmark[mp_idx]
        kps[coco_idx] = [lm.x * w, lm.y * h, lm.visibility]
    return kps


@dataclass
class PipelineResult:
    output_video: Path | None = None
    num_frames: int = 0
    keypoints_per_frame: list = field(default_factory=list)   # list of (17,3) arrays


def run_masked_piper_v2(
    input_path: Path,
    strategy: str = 'blur',
    draw_skeleton: bool = True,
    max_frames: int | None = None,
) -> PipelineResult:
    """Full Masked-Piper v2 pipeline (MediaPipe backend)."""
    if not HAS_MP:
        raise RuntimeError('MediaPipe required. pip install mediapipe')

    cap = cv2.VideoCapture(str(input_path))
    fps   = cap.get(cv2.CAP_PROP_FPS)
    W     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if max_frames:
        total = min(total, max_frames)

    out_path = input_path.parent / f'{input_path.stem}__v2_{strategy}.mp4'
    writer   = cv2.VideoWriter(str(out_path),
                                cv2.VideoWriter_fourcc(*'mp4v'), fps, (W, H))

    mp_pose = mp.solutions.pose
    mp_seg  = mp.solutions.selfie_segmentation
    fn      = STRATEGIES.get(strategy, apply_blur)
    kernel  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
    all_kps = []

    with mp_pose.Pose(model_complexity=1,
                      min_detection_confidence=0.5) as pose_est, \
         mp_seg.SelfieSegmentation(model_selection=1) as seg_est:

        for _ in tqdm(range(total), desc=f'Masked-Piper v2 [{strategy}]'):
            ok, frame = cap.read()
            if not ok:
                break
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            # 1. Segment
            conf = seg_est.process(rgb).segmentation_mask
            mask = (conf > 0.55).astype(np.uint8) * 255
            mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

            # 2. Mask
            out_frame = fn(frame, mask)

            # 3. Estimate pose & draw skeleton
            pose_res = pose_est.process(rgb)
            if pose_res.pose_landmarks:
                kps = mediapipe_to_coco17(pose_res.pose_landmarks, W, H)
                all_kps.append(kps)
                if draw_skeleton:
                    draw_coco17_skeleton(out_frame, kps)
            else:
                all_kps.append(np.zeros((17, 3)))

            writer.write(out_frame)

    cap.release()
    writer.release()
    print(f'Done → {out_path}')
    return PipelineResult(output_video=out_path,
                          num_frames=total,
                          keypoints_per_frame=all_kps)


print('run_masked_piper_v2() defined.')

In [ ]:
# Run the v2 pipeline on your clip (uncomment when INPUT_VIDEO is set)
# result = run_masked_piper_v2(INPUT_VIDEO, strategy='blur', max_frames=90)
# print(f'Processed {result.num_frames} frames')
# print(f'Keypoints shape: {np.array(result.keypoints_per_frame).shape}')
print('Uncomment and run when INPUT_VIDEO is ready.')

---
## Part 8 — Visualizing pose output

After processing, we can plot the keypoint trajectories (e.g. wrist position over time) to confirm that pose signal is preserved through masking.

In [ ]:
def plot_wrist_trajectory(keypoints_per_frame: list,
                           fps: float = 30.0) -> None:
    """
    Plot left and right wrist x-position over time.
    COCO-17 index 9 = L-wrist, 10 = R-wrist.
    """
    kps = np.array(keypoints_per_frame)  # (T, 17, 3)
    T   = kps.shape[0]
    t   = np.arange(T) / fps

    fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)

    for ax, idx, label, color in [
        (axes[0], 9,  'Left wrist x',  '#78a9ff'),
        (axes[1], 10, 'Right wrist x', '#42be65'),
    ]:
        conf = kps[:, idx, 2]
        x    = kps[:, idx, 0]
        x_masked = np.where(conf > 0.3, x, np.nan)
        ax.plot(t, x_masked, color=color, lw=1.5, label=label)
        ax.fill_between(t, x_masked, alpha=0.15, color=color)
        ax.set_ylabel('x position (px)', fontsize=9)
        ax.legend(fontsize=9, loc='upper right')
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel('Time (s)')
    plt.suptitle('Wrist trajectories — pose preserved through masking',
                 fontsize=11, color='#f4f4f4')
    plt.tight_layout()
    plt.show()


# ── Demo with synthetic random-walk trajectories ───────────────────
rng2 = np.random.default_rng(0)
T_demo = 150
demo_kps = np.zeros((T_demo, 17, 3))
for idx in [9, 10]:
    x_walk = np.cumsum(rng2.normal(0, 3, T_demo)) + 320
    demo_kps[:, idx, 0] = x_walk
    demo_kps[:, idx, 2] = 0.9   # high confidence

plot_wrist_trajectory(demo_kps, fps=30.0)

---
## Exercise

1. Change `STRATEGY` to `'pixelate'`, `'solid'`, or `'silhouette'` and re-run Part 5.
2. Try lowering `THRESHOLD` to `0.3` — how does the mask boundary change?
3. **Discussion**: Which strategy best preserves gesture motion visibility while protecting identity?

---
## Summary

| Component | Options | Trade-off |
|---|---|---|
| **Estimator** | MediaPipe, ViTPose | CPU vs GPU; accuracy vs speed |
| **Segmenter** | MediaPipe Selfie, SAM2-tiny, BBox | Quality vs latency |
| **Strategy** | blur, pixelate, solid, silhouette | Privacy level vs visual appeal |

The COCO-17 JSON output from Masked-Piper v2 feeds directly into MaskBench for quantitative evaluation — see **03_maskbench_evaluation.ipynb**.